# 포트홀 + 기본사물 YOLO 학습 (코랩)

yolo26n 하나로 **COCO 80가지 사물 + 포트홀(+우드락 등 내데이터)** 다 인식하게 학습함

그냥 포트홀만 학습시키면 원래알던 사람·차 같은거 까먹음 → 그래서 이렇게함
1. 포트홀 사진에 원래있던 사람·차 같은것도 큰모델(yolo26x)로 자동라벨 달아줌
2. COCO 사진 5000장(원래 80가지 정답있음)도 같이섞음
3. 그걸로 yolo26n 학습 → 포트홀은 새로배우고 원래사물은 안까먹음

**순서대로 위에서부터 실행하면됨.** 맨처음 `런타임 → 런타임 유형 변경 → T4 GPU` 꼭 켜기

## 0. GPU 확인 + 설치 (런타임 처음켤때 딱 한번만)
설치 끝나면 **런타임 → 세션 다시 시작** 한번 누르고 **1번칸부터** 실행하기
(안그러면 torch 두번불러와져서 `Only a single TORCH_LIBRARY` 오류남)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install ultralytics roboflow
print('설치끝. 이제 런타임 → 세션 다시 시작 → 1번칸부터')

## 1. 구글드라이브 연결
학습결과를 드라이브에 저장함 → 코랩 끊겨도 이어서 학습가능

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE = '/content/drive/MyDrive/pothole_yolo'   # 결과저장할 폴더 (바꿔도됨)
os.makedirs(SAVE, exist_ok=True)
print('저장폴더:', SAVE)

## 2. 포트홀 데이터 받기 (둘중 하나)

**A. Roboflow (추천)**
1. https://universe.roboflow.com 가서 `pothole` 검색 → 사진 많고 박스(Object Detection)로 된거 고르기
2. `Dataset` → `Download Dataset` → 형식 **YOLOv8** → `show download code` 누르면 코드나옴
3. 그코드에 있는 `api_key`, `workspace`, `project`, `version` 을 아래에 넣기 (가입 무료)

**B. 내가 가진 zip** (Kaggle 등에서 받은 YOLO형식) → 드라이브에 올리고 `ZIP_PATH` 에 경로넣기

In [ ]:
USE = 'roboflow'          # 'roboflow' 또는 'zip'

# --- A. Roboflow ---
RF_API_KEY   = ''         # 여기에 키
RF_WORKSPACE = ''         # 예: download code 의 rf.workspace("여기")
RF_PROJECT   = ''         # rf.workspace(..).project("여기")
RF_VERSION   = 1          # .version(여기)

# --- B. zip ---
ZIP_PATH = '/content/drive/MyDrive/pothole.zip'

if USE == 'roboflow':
    from roboflow import Roboflow
    rf = Roboflow(api_key=RF_API_KEY)
    ds = rf.workspace(RF_WORKSPACE).project(RF_PROJECT).version(RF_VERSION).download('yolov8', location='/content/pothole_src')
    POT_SRC = ds.location
else:
    !rm -rf /content/pothole_src && mkdir -p /content/pothole_src
    !unzip -q "$ZIP_PATH" -d /content/pothole_src
    POT_SRC = '/content/pothole_src'
print('포트홀데이터:', POT_SRC)
!find "$POT_SRC" -maxdepth 2 -type d | head -20

## 2-1. (선택) 내 데이터 추가 — 우드락 등
우드락 사진 찍어서 Roboflow 에서 박스치고(Annotate) YOLOv8 로 내보낸 zip 을 드라이브에 올리기
- 라벨이름에 `pot` 들어가면 포트홀로 침
- 나머지 이름(예: `woodlock`)은 새 종류로 추가됨
- 없으면 그냥 빈칸으로 두고 넘어가기

In [ ]:
MY_ZIP = ''   # 예: '/content/drive/MyDrive/woodlock.zip'  없으면 ''

MY_SRC = None
if MY_ZIP:
    !rm -rf /content/my_src && mkdir -p /content/my_src
    !unzip -q "$MY_ZIP" -d /content/my_src
    MY_SRC = '/content/my_src'
print('내데이터:', MY_SRC)

## 3. COCO 5000장 받기 (원래 80가지 안까먹게)
val2017 사진 5000장 → 4000장 학습, 1000장 검사에 씀. 1GB쯤이라 몇분걸림

In [ ]:
%cd /content
!wget -q -nc http://images.cocodataset.org/zips/val2017.zip
!wget -q -nc https://github.com/ultralytics/assets/releases/download/v0.0.0/coco2017labels.zip
!unzip -q -n val2017.zip -d /content/coco_img
!unzip -q -n coco2017labels.zip -d /content
COCO_IMG = '/content/coco_img/val2017'
COCO_LBL = '/content/coco/labels/val2017'
import glob
print('COCO 사진', len(glob.glob(COCO_IMG + '/*.jpg')), '장, 라벨', len(glob.glob(COCO_LBL + '/*.txt')), '개')

## 4-A. 데이터 합치기 (금방끝남)
- 0~79: COCO 원래사물 / 80: pothole / 81~: 다른종류 (맨홀 등)
- 사진 복사만함. 메모리 거의안씀

In [ ]:
import os, glob, random, shutil, yaml, json
from pathlib import Path

# ===== 앞칸 안돌렸어도(세션 재시작돼도) 되게: 데이터있는곳 직접찾음 =====
POT_SRC = globals().get('POT_SRC') or '/content/pothole_src'
if not os.path.isdir(POT_SRC):  # Roboflow 가 다른이름으로 받았으면 data.yaml 있는폴더 찾기
    cands = [p.parent for p in Path('/content').glob('*/data.yaml') if 'merged' not in str(p)]
    if cands:
        POT_SRC = str(cands[0])
MY_SRC   = globals().get('MY_SRC')   # 2-1칸 건너뛰었어도 괜찮게
COCO_IMG = globals().get('COCO_IMG') or '/content/coco_img/val2017'
COCO_LBL = globals().get('COCO_LBL') or '/content/coco/labels/val2017'
missing = [p for p in (POT_SRC, COCO_IMG, COCO_LBL) if not os.path.isdir(p)]
if missing:
    raise RuntimeError(f'이폴더없음: {missing} → 런타임이 완전초기화된거. 2번(포트홀), 3번(COCO)칸 다시 실행하기')
print('포트홀:', POT_SRC, '| COCO:', COCO_IMG)

OUT = Path('/content/merged')
shutil.rmtree(OUT, ignore_errors=True)
for s in ('train', 'val'):
    (OUT / 'images' / s).mkdir(parents=True)
    (OUT / 'labels' / s).mkdir(parents=True)

COCO_NAMES = yaml.safe_load(open(Path(__import__('ultralytics').__file__).parent / 'cfg/datasets/coco.yaml'))['names']
names = [COCO_NAMES[i] for i in range(80)] + ['pothole']
POT = 80
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

def find_pairs(root):
    # 데이터폴더 안에서 (사진, 라벨txt) 짝 다찾음. 폴더구조 달라도 images↔labels 로 맞춤
    pairs = []
    for img in Path(root).rglob('*'):
        if img.suffix.lower() not in IMG_EXT:
            continue
        lbl = Path(str(img.parent).replace('images', 'labels')) / (img.stem + '.txt')
        pairs.append((img, lbl if lbl.exists() else None))
    return pairs

def read_names(root):
    # data.yaml 에서 라벨이름 읽음
    for y in Path(root).rglob('data.yaml'):
        d = yaml.safe_load(open(y))
        n = d.get('names')
        return list(n.values()) if isinstance(n, dict) else list(n)
    return ['pothole']

def class_map(src_names):
    # 원래번호 → 새번호. pot 들어가면 포트홀, 나머진 새종류로 추가
    m = {}
    for i, n in enumerate(src_names):
        if 'pot' in n.lower():
            m[i] = POT
        else:
            if n not in names:
                names.append(n)
            m[i] = names.index(n)
    return m

def copy_set(pairs, cmap, tag, split_val=0.15, seed=0):
    random.Random(seed).shuffle(pairs)
    nval = int(len(pairs) * split_val)
    done = []
    for k, (img, lbl) in enumerate(pairs):
        s = 'val' if k < nval else 'train'
        new = f'{tag}_{k:05d}'
        dst_img = OUT / 'images' / s / (new + img.suffix.lower())
        shutil.copy(img, dst_img)
        lines = []
        if lbl:
            for line in open(lbl).read().split('\n'):
                p = line.split()
                if len(p) >= 5 and int(p[0]) in cmap:
                    lines.append(' '.join([str(cmap[int(p[0])])] + p[1:]))
        (OUT / 'labels' / s / (new + '.txt')).write_text('\n'.join(lines) + ('\n' if lines else ''))
        done.append(str(dst_img))
    return done

pot_names = read_names(POT_SRC)
print('포트홀데이터 라벨이름:', pot_names)
MAX_POT = 5000   # 포트홀사진 몇장만쓸지 (전부쓰려면 None)
pot_pairs = find_pairs(POT_SRC)
random.Random(0).shuffle(pot_pairs)
pot_pairs = pot_pairs[:MAX_POT] if MAX_POT else pot_pairs
print('포트홀사진', len(pot_pairs), '장 씀')
todo = copy_set(pot_pairs, class_map(pot_names), 'pot')
if MY_SRC:
    my_names = read_names(MY_SRC)
    print('내데이터 라벨이름:', my_names)
    todo += copy_set(find_pairs(MY_SRC), class_map(my_names), 'my')

coco_pairs = [(Path(p), Path(COCO_LBL) / (Path(p).stem + '.txt')) for p in sorted(glob.glob(COCO_IMG + '/*.jpg'))]
coco_pairs = [(i, l if l.exists() else None) for i, l in coco_pairs]
copy_set(coco_pairs, {i: i for i in range(80)}, 'coco', split_val=0.2)

json.dump(todo, open(OUT / '_autolabel_todo.json', 'w'))   # 4-B에서 자동라벨 달 사진목록
DATA = '/content/merged/data.yaml'
yaml.safe_dump({'path': str(OUT), 'train': 'images/train', 'val': 'images/val',
                'names': {i: n for i, n in enumerate(names)}}, open(DATA, 'w'), allow_unicode=True)
for s in ('train', 'val'):
    print(s, len(list((OUT / 'images' / s).iterdir())), '장')
print('종류', len(names), '개:', names[80:])
print('자동라벨 달사진', len(todo), '장 → 4-B 실행')

## 4-B. 포트홀사진에 원래사물 자동라벨 (오래걸림, 끊겨도 이어서함)
- 포트홀사진 속 사람·차 등을 큰모델(yolo26x)로 찾아서 라벨추가 (신뢰도 0.5이상만)
- **200장씩** 끊어서하고 메모리 비움. 어디까지했는지 기록함
- 또 RAM 다운되면: 그냥 **이칸만 다시실행** (4-A 다시하면 처음부터라 안됨)
- 그래도 계속 다운되면 아래 `TEACHER` 를 `'yolo26l.pt'` 로 (조금 작은모델)
- `CUDA out of memory` 뜨면: 세션 다시시작 → `batch=4` 를 `batch=1` 로 → 이칸만 다시실행

In [ ]:
import json, gc, torch
from pathlib import Path
from ultralytics import YOLO

TEACHER = 'yolo26x.pt'   # 계속 메모리부족이면 'yolo26l.pt' 로
CHUNK = 200              # 200장씩 끊어서함
BATCH = 8                # 한번에 넣는장수. GPU메모리 모자라면 알아서 반으로줄임
OUT = Path('/content/merged')
todo = json.load(open(OUT / '_autolabel_todo.json'))
done_f = OUT / '_autolabel_done.txt'
done = set(done_f.read_text().split('\n')) if done_f.exists() else set()
left = [p for p in todo if p not in done]
print(f'전체 {len(todo)}장, 이미함 {len(todo) - len(left)}장, 남은거 {len(left)}장')

gc.collect(); torch.cuda.empty_cache()   # 전에 터진 메모리 비우기
teacher = YOLO(TEACHER)
chunk_txt = OUT / '_chunk.txt'           # ★목록을 txt로 넘겨야 batch대로 나눠서함 (list로 넘기면 200장 한번에올라감)
added = 0
for i in range(0, len(left), CHUNK):
    part = left[i:i + CHUNK]
    chunk_txt.write_text('\n'.join(part) + '\n')
    while True:
        try:
            new_lines = {}   # 이번 200장 결과 모아뒀다가 다끝나면 한번에 씀 (중간에터져도 중복안생김)
            for r in teacher.predict(str(chunk_txt), conf=0.5, stream=True, verbose=False, batch=BATCH, imgsz=640):
                if len(r.boxes):
                    lbl = Path(str(Path(r.path).parent).replace('images', 'labels')) / (Path(r.path).stem + '.txt')
                    new_lines[lbl] = [f'{int(c)} ' + ' '.join(f'{v:.6f}' for v in b)
                                      for c, b in zip(r.boxes.cls.tolist(), r.boxes.xywhn.tolist())]
                del r
            break
        except torch.cuda.OutOfMemoryError:
            gc.collect(); torch.cuda.empty_cache()
            if BATCH == 1:
                raise RuntimeError('batch=1 인데도 메모리부족 → 세션 다시시작하고 TEACHER 를 yolo26l.pt 로')
            BATCH = max(1, BATCH // 2)
            print(f'  GPU메모리 부족 → batch {BATCH} 로 줄여서 다시함')
    for lbl, lines in new_lines.items():
        with open(lbl, 'a') as f:
            f.write('\n'.join(lines) + '\n')
        added += len(lines)
    with open(done_f, 'a') as f:            # 여기까지했다고 기록
        f.write('\n'.join(part) + '\n')
    gc.collect(); torch.cuda.empty_cache()   # 메모리비움
    print(f'  {min(i + CHUNK, len(left))}/{len(left)}  (박스 {added}개 추가, batch {BATCH})')
print('자동라벨 끝! 추가한박스:', added)
DATA = '/content/merged/data.yaml'

## 5. 학습
T4 에서 한 에폭 1~2분쯤 → 60에폭이면 1~2시간. 결과는 드라이브에 저장됨
- 중간에 끊기면 아래 **5-1** 실행하면 이어서함

In [ ]:
from ultralytics import YOLO
SAVE = globals().get('SAVE', '/content/drive/MyDrive/pothole_yolo')
DATA = globals().get('DATA', '/content/merged/data.yaml')
model = YOLO('yolo26n.pt')          # COCO로 미리배운거에서 시작
model.train(data=DATA, epochs=60, imgsz=640, batch=32, patience=15,
            project=SAVE, name='pothole_coco', exist_ok=True, workers=2)

In [ ]:
# 5-1. 끊겼을때 이어서 학습
# model = YOLO(f'{SAVE}/pothole_coco/weights/last.pt'); model.train(resume=True)

## 6. 얼마나 잘하나 확인
- **포트홀 mAP50**: 포트홀 얼마나 잘찾는지
- **원래사물**: 새모델 vs 원래 yolo26n (같은 COCO 검사사진 1000장) → 안까먹었는지

In [ ]:
import os, shutil, yaml
from pathlib import Path
from ultralytics import YOLO
SAVE = globals().get('SAVE', '/content/drive/MyDrive/pothole_yolo')   # 재시작돼도 되게
DATA = '/content/merged/data.yaml'
OUT = Path('/content/merged')
names = [n for _, n in sorted(yaml.safe_load(open(DATA))['names'].items())]
POT = 80
COCO_NAMES = {i: names[i] for i in range(80)}
BEST = f'{SAVE}/pothole_coco/weights/best.pt'
new = YOLO(BEST)
m = new.val(data=DATA, imgsz=640, verbose=False)
maps50 = {}
for i, ci in enumerate(m.box.ap_class_index):
    maps50[int(ci)] = float(m.box.ap50[i])
print(f"포트홀 mAP50 = {maps50.get(POT, float('nan')):.3f}")
for i in range(81, len(names)):
    print(f"{names[i]} mAP50 = {maps50.get(i, float('nan')):.3f}")

# 원래사물 비교: COCO 검사사진만으로
coco_val = Path('/content/coco_val_only'); shutil.rmtree(coco_val, ignore_errors=True)
for d in ('images', 'labels'):
    (coco_val / d).mkdir(parents=True)
for p in (OUT / 'images' / 'val').glob('coco_*'):
    os.symlink(p, coco_val / 'images' / p.name)
    os.symlink(OUT / 'labels' / 'val' / (p.stem + '.txt'), coco_val / 'labels' / (p.stem + '.txt'))
yaml.safe_dump({'path': str(coco_val), 'train': 'images', 'val': 'images',
                'names': {i: COCO_NAMES[i] for i in range(80)}}, open('/content/coco_val.yaml', 'w'))
old_m = YOLO('yolo26n.pt').val(data='/content/coco_val.yaml', imgsz=640, verbose=False)
new_coco = [float(m.box.ap50[i]) for i, ci in enumerate(m.box.ap_class_index) if int(ci) < 80]
print(f"원래사물 mAP50: 원래 yolo26n {old_m.box.map50:.3f}  →  새모델 {sum(new_coco)/len(new_coco):.3f}")

## 7. 사진 하나로 눈으로 확인

In [ ]:
from IPython.display import Image, display
sample = sorted((OUT / 'images' / 'val').glob('pot_*'))[:3]
for r in new.predict([str(s) for s in sample], conf=0.25, save=True, project='/content/pred', name='x', exist_ok=True, verbose=False):
    pass
for f in sorted(Path('/content/pred/x').glob('*'))[:3]:
    display(Image(filename=str(f), width=500))

## 8. 가볍게 내보내기 (경량화)
- **NCNN**: 라즈베리파이 CPU에서 제일빠름 (Ultralytics 추천)
- **TFLite int8**: 안드로이드·블랙박스칩용, 크기 제일작음
- **ONNX**: 아무데서나

In [ ]:
outs = {}
for fmt, kw in (('ncnn', {}), ('tflite_int8', dict(format='tflite', int8=True, data=DATA)), ('onnx', dict(format='onnx', simplify=True))):
    try:
        outs[fmt] = new.export(**(kw or dict(format='ncnn')), imgsz=640)
    except Exception as e:
        print(fmt, '내보내기 실패 (다른건 계속함):', e)

def size_mb(p):
    p = Path(p)
    if p.is_dir():
        return sum(f.stat().st_size for f in p.rglob('*') if f.is_file()) / 1e6
    return p.stat().st_size / 1e6
print(f"best.pt      {size_mb(BEST):6.1f} MB")
for k, p in outs.items():
    print(f"{k:12s} {size_mb(p):6.1f} MB  {p}")

# 파이로 옮길거 zip 으로 드라이브에저장 (내보낸파일은 weights 폴더안에 생김)
!cd "{SAVE}/pothole_coco/weights" && zip -qr "{SAVE}/best_ncnn_model.zip" best_ncnn_model
print('드라이브 pothole_yolo/best_ncnn_model.zip ← 이거 파이로 옮기면됨')

## 9. 내 컴퓨터로 바로 다운로드 (다운로드 폴더에 저장됨)
처음이면 '여러 파일 다운로드 허용' 눌러주기. 받는동안 탭 닫지않기

In [ ]:
from google.colab import files
import shutil
SAVE = '/content/drive/MyDrive/pothole_yolo'
w = f'{SAVE}/pothole_coco/weights'
shutil.make_archive('/content/best_ncnn_model', 'zip', w, 'best_ncnn_model')   # 파이용 모델 zip으로묶기
files.download('/content/best_ncnn_model.zip')   # 파이용 (NCNN)
files.download(f'{w}/best.pt')                   # 원본모델 (혹시몰라서)